# Building a Tiered, Tool-Using Finance Agent with LangChain + LangGraph

This notebook walks through **your `finance_agent.py`** line by line, but instead of just showing you the code, it *rebuilds it with you*, one concept at a time.

**Why this script is a good teaching example.** It isn't a toy "hello world" agent. In ~250 lines it touches almost every piece of the modern LangChain agent stack:

| Concept | Where it shows up |
|---|---|
| Provider-agnostic model loading | `init_chat_model` |
| Typed, injected runtime context | `ToolRuntime[UserContext]` |
| Function calling / tools | `@tool` |
| Structured (schema-validated) output | `pydantic` + `ToolStrategy` |
| Cross-cutting behavior (middleware) | `wrap_model_call`, `dynamic_prompt`, `wrap_tool_call` |
| Conversation memory | `InMemorySaver` (a **checkpointer**) |
| Multi-tenant / tiered behavior | `UserContext.membership_tier` driving both the **model** and the **prompt** |

**How we'll build it.** We won't dump the whole file at once. We'll start from an empty script and add exactly one new idea per section — imports only when we first need them, functions only when the previous ones are in place — so that by the end you have re-derived the entire file and understand *why* each piece exists, not just *that* it exists.

**A note on bugs.** Your script has a handful of small real bugs and rough edges (a self-assignment, a typo in an account name, an unused variable, a missing `thread_id`, etc.). I'm **not** silently fixing these — I'm flagging them as we pass them, because spotting this kind of thing is exactly the skill that makes debugging LangChain agents tractable. Look for the ⚠️ **Bug watch** callouts.

Let's start.

## 0. The mental model before we write any code

An "agent" in LangChain/LangGraph is really just:

```
loop:
    1. Send the conversation so far to an LLM, along with a list of tools it's allowed to call
    2. The LLM replies with either:
          a) a normal text answer  -> done, return it
          b) a request to call one or more tools -> run those Python functions,
             feed the results back into the conversation, go to step 1
```

Everything in this script is decoration around that loop:
- **Tools** = the Python functions the LLM is allowed to call (steps you don't want the LLM to "hallucinate", you want it to actually *execute*, e.g. reading a real balance).
- **Context** = data about *who* is asking, injected outside the LLM's control (so the LLM can't be tricked into acting as a different user).
- **Middleware** = code that runs *around* the loop's steps (before/after a model call, before/after a tool call, or to build the prompt dynamically) — this is how you get per-user behavior without writing five separate agents.
- **Structured output** = forcing the LLM's *final* answer to match a schema (a Pydantic model) instead of free text, so your application code can reliably read `.summary`, `.action_items`, etc.
- **Checkpointer** = a place to persist conversation state between calls, so the agent "remembers" earlier turns.

Keep this loop in your head — every section below is just filling in one box of it.

## 1. Environment setup — keeping secrets out of your code

Your agent needs an API key (for Groq) to talk to an LLM. The one thing you must never do is write `api_key = "gsk_abc123..."` directly in a `.py` file — it will end up in git history, in screenshots, in logs.

The standard fix is a **`.env` file** (never committed to git) plus the `python-dotenv` package to load it into `os.environ` at runtime.

In [ ]:
from dotenv import load_dotenv
import os

# load_dotenv() reads a file named `.env` in the current directory (or a parent
# directory) and injects every KEY=VALUE line into os.environ, as if you had
# run `export KEY=VALUE` in your shell before starting Python.
load_dotenv()

# Your .env file should look like:
#   GROQ_API_KEY=gsk_your_real_key_here
#
# and now this line can read it safely:
api_key = os.getenv("GROQ_API_KEY")
print("Key loaded:", bool(api_key))

**Why `os.getenv` and not `os.environ["GROQ_API_KEY"]`?** `.getenv(key)` returns `None` if the variable is missing instead of raising a `KeyError`. That's usually what you want during development — you get a clear `False`/`None` you can check, instead of a crash with a less obvious traceback.

> ⚠️ **Note:** in your original script `load_dotenv()` is called **twice** (once near the top, once again right before the tool definitions, around line 129). It's harmless — calling it twice just re-reads the same file — but it's a sign the file grew by copy-pasting blocks rather than a single top-to-bottom pass. Good to know when you're skimming your own code later.

## 2. Talking to an LLM — `init_chat_model` vs a provider-specific class

There are two ways to get a "chat model" object in LangChain, and your script actually uses **both**, which is itself worth understanding.

### 2.1 `init_chat_model` — the provider-agnostic factory

`init_chat_model` is a **factory function**: you give it a model identifier as a string, and it returns a ready-to-use chat model object, regardless of which company makes that model. This matters because LangChain supports dozens of providers (OpenAI, Anthropic, Groq, Google, ...) and each has its own SDK class (`ChatOpenAI`, `ChatGroq`, `ChatGoogleGenerativeAI`, ...). `init_chat_model` lets your *application code* stay provider-agnostic — you can swap `"groq:llama-3.1-8b-instant"` for `"openai:gpt-4o-mini"` and nothing else in your code has to change.

The string format is `"provider:model-name"`.

In [ ]:
from langchain.chat_models import init_chat_model

# temperature: 0.0-2.0, controls randomness of sampling.
#   - low (0-0.3)  -> deterministic, focused, good for factual/financial tasks
#   - high (0.7+)  -> more varied/creative, worse for tasks needing precision
# For a finance assistant you want it fairly low so numbers/advice don't drift.

# max_tokens: hard cap on how long the *reply* can be (not the whole conversation).
# This directly controls cost and latency — bigger cap = potentially slower/pricier replies.

basic_model = init_chat_model(
    'groq:llama-3.1-8b-instant',
    temperature=0.5,
    max_tokens=512
)

premium_model = init_chat_model(
    'groq:llama-3.3-70b-versatile',
    temperature=0.5,
    max_tokens=2028
)

platinum_model = init_chat_model(
    'groq:llama-3.3-70b-versatile',
    temperature=0.5
    # no max_tokens set -> falls back to the provider's default ceiling,
    # i.e. platinum users are deliberately allowed longer, more detailed replies
)

**Why three separate model objects instead of one?** This is the setup for a pattern you'll meet again in the middleware section: **route different users to different models based on a business property (their membership tier)**. `basic_model` uses a small, fast, cheap 8B-parameter model; `premium_model` and `platinum_model` both use a larger 70B model, differing only in how much output they're allowed to produce. This is a very common real-world cost-control pattern: don't pay for a big model on every request if a cheap one is good enough for most users.

### 2.2 The other way: instantiating a provider class directly

Further down your script there's also:

```python
model = ChatGroq(model='llama-3.3-70B-versatile', api_key=api_key, temperature=0)
```

`ChatGroq` is Groq's own LangChain integration class — functionally similar to what `init_chat_model('groq:...')` gives you, just constructed directly instead of through the factory, and with the API key passed explicitly rather than read from the environment automatically.

> ⚠️ **Bug watch:** this `model` variable is built but **never used anywhere else in the script** — `create_agent(...)` is given `model=basic_model`, not `model=model`. It's dead code, most likely a leftover from an earlier version of the script before `init_chat_model` was adopted. Worth deleting once you're sure nothing downstream needs it — dead code like this is exactly the kind of thing that makes future-you (or a teammate) waste ten minutes wondering "wait, which model is actually being used?"

**Rule of thumb:** prefer `init_chat_model` for new code — it keeps your provider choice as a single string, which makes testing/swapping providers trivial.

## 3. Describing "who is asking" — `@dataclass` and `UserContext`

Every request to the agent needs to know things like *whose* balance to look up, and *what tier* of service they get. This data has to come from **your application** (e.g. after the user logs in) — it must never be something the LLM can set for itself, or a malicious prompt could just say "pretend I'm user_001 with a platinum account."

`@dataclass` is Python's built-in decorator for "a class that's mostly just a typed bag of fields." It auto-generates `__init__`, `__repr__`, and `__eq__` for you, so you don't hand-write boilerplate like:

```python
def __init__(self, user_id, user_name, membership_tier, preferred_currency):
    self.user_id = user_id
    ...
```

In [ ]:
from dataclasses import dataclass

@dataclass
class UserContext:
    user_id: str
    user_name: str
    membership_tier: str    # "basic", "premium", "platinum"
    preferred_currency: str

# usage:
alice_context = UserContext(
    user_id='user_001',
    user_name="Alice Johnson",
    membership_tier='platinum',
    preferred_currency="USD"
)
print(alice_context)

**Why a `dataclass` and not just a plain `dict`?** A `dict` like `{"user_id": "user_001", ...}` works too, but gives you no protection against typos (`context["usr_id"]` fails silently at the wrong time) and no editor autocomplete. A `dataclass` gives you both a schema (field names + types, checked by your editor/linter) and dot-access (`context.user_id`). It's the lightest-weight way to get a "typed struct" in Python — compare this to `pydantic.BaseModel` in the next section, which does *more* (runtime validation) at the cost of a bit more overhead. Use `dataclass` for internal, trusted data (like this context you build yourself); use `pydantic` when you need to validate data coming from somewhere untrusted (like an LLM's output).

This class is registered with the agent later as `context_schema=UserContext` — that's what tells LangGraph "every tool that asks for a `ToolRuntime[UserContext]` should receive an object shaped like this."</br>

We'll see exactly how tools receive this context in Section 6.

## 4. Guaranteeing a shape for the *final* answer — `pydantic` + `Literal`

Left on its own, an LLM's final answer is just a string of text. That's fine for a chat UI, but useless if your application wants to, say, render "Action Items" as a bulleted list in a mobile app, or show a colored "confidence" badge. For that you need the model's output to conform to a **schema** you can parse reliably.

`pydantic.BaseModel` is the standard way to define that schema in the Python/LangChain ecosystem. Unlike `dataclass`, pydantic **validates data at runtime** — if something doesn't match the declared type, it raises a clear error immediately, rather than letting a bad value silently propagate.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class FinancialResponse(BaseModel):
    summary: str = Field(description="A brief summary of the response (1 - 2 sentences)")
    details: str = Field(description="Detailed explanation or data")

    action_items: list[str] = Field(
        default_factory=list,
        description="List of recommended actions the user should take"
    )

    warning: list[str] = Field(
        default_factory=list,
        description="Any warnings or concerns to highlight"
    )

    confidence: Literal["high", "medium", "lower"] = Field(
        default="high",
        description="Confidence level in the advice provided"
    )

A few details worth understanding:

- **`Field(description=...)`** isn't just documentation for humans. When this schema is handed to the LLM (via `ToolStrategy`, in Section 8), the description text is literally what tells the model *what to put in each field*. Write these descriptions like you're briefing a new employee, not like a code comment — the LLM only knows what the description tells it.
- **`default_factory=list`** vs **`default=[]`**: never use a mutable object like `[]` directly as a default value in Python — every instance would then *share the same list* (a classic Python gotcha). `default_factory` tells pydantic "call `list()` fresh for every new instance" instead.
- **`Literal["high", "medium", "lower"]`** is a type that can *only* be one of those exact strings. This is stricter than `str` — it means `confidence` can never accidentally end up as `"very high"` or `"HIGH"`; pydantic will reject anything outside the three allowed values. This is how you turn "the LLM should pick one of these options" into something your code can `if/elif` on safely.

  (Small English nit, not a bug: `"lower"` is a slightly odd choice next to `"high"`/`"medium"` — `"low"` would read more naturally as the third rung of the scale. Doesn't affect functionality at all, just worth noticing since it's *your* schema and the LLM will use these exact words.)

## 5. A stand-in for a real backend — `USER_DATABASE`

Real agents call real APIs/databases. For learning purposes (and for anyone reading this notebook without your production backend), the script uses a plain nested `dict` as an in-memory fake database. This is a good practice when developing a tool-using agent: **build and test the agent's reasoning against a fake data source first**, then swap the fake for a real API call later — the tool *function signatures* don't need to change, only their internals.

In [ ]:
USER_DATABASE = {
    "user_001": {
        "name": "Alice Johnson",
        "accounts": {
            "checking": 2500.00,
            "savings": 15000.00,
            "investment": 45000.00,
        },
        "transactions": {
            "checking": [
                {"date": "2025-01-15", "description": "Grocery Store", "amount": -85.50},
                {"date": "2025-01-14", "description": "Direct Deposit", "amount": 3200.00},
                {"date": "2025-01-13", "description": "Electric Bill", "amount": -120.00},
                {"date": "2025-01-12", "description": "Restaurant", "amount": -45.00},
                {"date": "2025-01-10", "description": "Gas Station", "amount": -55.00},
            ],
            "savings": [
                {"date": "2025-01-01", "description": "Interest Payment", "amount": 12.50},
                {"date": "2024-12-15", "description": "Transfer from Checking", "amount": 500.00},
            ],
            "investment": [
                {"date": "2025-01-14", "description": "Dividend - AAPL", "amount": 125.00},
                {"date": "2025-01-10", "description": "Buy - VTI", "amount": -1000.00},
            ],
        },
    },
    "user_002": {
        "name": "Bob Smith",
        "accounts": {
            "checking": 1200.00,
            "savings": 8000.00,
            "investment": 22000.00,
        },
        "transactions": {
            "checking": [
                {"date": "2025-01-15", "description": "Coffee Shop", "amount": -5.50},
                {"date": "2025-01-14", "description": "Freelance Payment", "amount": 1500.00},
            ],
            "savings": [
                {"date": "2025-01-01", "description": "Interest Payment", "amount": 6.50},
            ],
            "investment": [
                {"date": "2025-01-12", "description": "Dividend - VTI", "amount": 45.00},
            ],
        },
    },
}

## 6. Tools — giving the LLM hands

### 6.1 What is a "tool", really?

Under the hood, LLM providers support "function calling": you send the model a list of function *signatures* (name, description, parameters) alongside the conversation, and the model can respond with "please call `get_account_balance` with `account_type='checking'`" instead of a normal text reply. Your code then actually runs that Python function, and feeds the return value back to the model as if it were part of the conversation.

The model **never executes your code itself** — it only ever *requests* a call. This is exactly the sandboxing you want: the LLM can't do anything you haven't explicitly given it a function for.

### 6.2 The `@tool` decorator

LangChain's `@tool` decorator turns an ordinary Python function into something the agent can offer to the LLM. It automatically extracts:
- the function **name** → becomes the tool's name the LLM sees
- the **docstring** → becomes the tool's description (this is *the* most important part — the LLM decides *when* to call a tool almost entirely based on this text, so vague docstrings cause tools to be called at the wrong time or not at all)
- the **type-annotated parameters** → become the tool's input schema (so the LLM knows what arguments it must supply, and in what shape)

Let's build the simplest tool in the file first.

In [ ]:
from langchain.tools import tool

@tool
def calculate_budget(monthly_income: float, expense_category: str) -> str:
    """
        Calculate recommended budget allocation for an expense category.

        Args:
            monthly_income : float - User's monthly income
            expense_category: str - Expense category the user is interested in
    """
    allocations = {
        'housing': 0.30,
        'food': 0.12,
        'transportation': 0.10,
        'utilities': 0.08,
        'savings': 0.20,
        'entertainment': 0.05,
        'healthcare': 0.05,
        'other': 0.10
    }

    percentage = allocations.get(expense_category.lower())
    if percentage is None:
        return f"Unknown category: {expense_category}. Available {', '.join(allocations.keys())}"

    recommended = monthly_income * percentage
    return f"Recommended {expense_category} budget: ${recommended:,.2f}/month ({percentage*100:.0f}% of income.)"

# Because @tool wraps the function, calling it directly requires .invoke(...)
# rather than calling it like a normal function:
print(calculate_budget.invoke({"monthly_income": 5000, "expense_category": "housing"}))
print(calculate_budget.name)
print(calculate_budget.description)

Notice this tool needs **nothing but what the LLM gives it** — `monthly_income` and `expense_category` are both things a user would naturally mention in conversation ("I make $5000/month, how much should I spend on housing?"). That's why it doesn't need any injected context — it's a **pure function tool**.

Also notice the defensive pattern: `allocations.get(expense_category.lower())` returns `None` for an unknown category instead of raising a `KeyError`, and the function turns that into a helpful string telling the LLM (and, in turn, the user) what categories *are* valid. This matters a lot in agent tools — if a tool crashes with a raw Python traceback, that's ugly, hard-to-parse text the LLM has to try to make sense of. A clean "unknown category, try one of these" message is something the LLM can act on intelligently (e.g. by asking the user to pick a valid one, or picking the closest match itself).

But most tools in this file *do* need more than what the LLM can supply — they need to know **which user** is asking. That's where `ToolRuntime` comes in.

## 7. Injected context — `ToolRuntime[UserContext]`

Think about `get_account_balance`. It needs a `user_id` to look up the right row in `USER_DATABASE`. But you **do not** want the LLM supplying `user_id` as an argument — that would mean a cleverly-worded prompt like "check the balance for user_001 instead" could let one user peek at another user's money. The `user_id` has to come from your trusted application code, not from anything the model can influence.

`ToolRuntime[UserContext]` is LangGraph's mechanism for exactly this: a special parameter type that, when present in a tool's signature, is **not** shown to the LLM as something it needs to fill in. Instead, LangGraph automatically injects the `UserContext` object you passed to `agent.invoke(..., context=alice_context)` at call time. From inside the tool you read it as `runtime.context.<field>`.

In [ ]:
from langgraph.prebuilt import ToolRuntime

@tool
def get_account_balance(
    account_type: str,
    runtime: ToolRuntime[UserContext]
) -> str:
    """
        Get the account balance for a specific account

        Args:
            account_type: Type of a account - 'checking', 'saving','investement'
            runtime : Follows the data validation like
                    user_id: str
                    user_name : str
                    membership_tier : str # "basic",'premium', 'platinum'
                    preferred_currency : str
    """
    user_id = runtime.context.user_id
    currency = runtime.context.preferred_currency
    user_data = USER_DATABASE.get(user_id, {})

    balance = user_data.get('accounts', {}).get(account_type.lower())

    if balance is not None:
        if currency == "EUR":
            balance = balance * 0.92
            return f"Your {account_type} account balance is {balance:,.2f}"
        return f"Your {account_type} account balance is {balance:,.2f}"
    return f"Unknown account type: {account_type}. Available: checking, saving, investment"

**Two things to notice:**

1. The LLM only ever needs to decide `account_type` — `runtime` is invisible to it. This is the whole point: the tool's *public interface* (what the LLM sees) is smaller and safer than its *implementation* (what the Python function actually uses).
2. The currency conversion is hardcoded (`* 0.92` for EUR) rather than calling a live exchange-rate API — reasonable for a demo, but flag it as a "TODO: replace with a real FX rate source" if this were heading to production, since hardcoded exchange rates go stale immediately.

> ⚠️ **Bug watch — an actual functional bug, not just style:** the docstring and the returned error message both say the valid account types are `'checking'`, `'saving'` (no *s*) and `'investement'` (misspelled) / `'investment'`. But `USER_DATABASE`'s actual keys are `"checking"`, `"savings"` (with an *s*), and `"investment"`. So if the LLM ever reads the docstring literally and asks for `account_type="saving"`, `.get("saving")` on the real dict will return `None` — a silent, confusing failure, blamed on "unknown account type" even though the user's intent was completely valid. This is a great example of why **docstring accuracy isn't just style — the LLM treats your docstring as ground truth**, so a typo there is a bug, not a nitpick. Worth fixing to `'savings'`.

Now let's build the next tool, which follows the exact same pattern but also takes a normal (LLM-supplied) argument *alongside* the injected runtime.

In [ ]:
@tool
def get_recent_transactions(
    account_type: str,
    runtime: ToolRuntime[UserContext],
    limit: int = 5
) -> str:
    """
        Get recent transactions for an account

        Args:
            account_type: Type of account - savings, investment, checking
            limit: int - Number of recent transactions needed to be showed - default = 5
    """
    user_id = runtime.context.user_id
    user_data = USER_DATABASE.get(user_id, {})

    account_transactions = user_data.get("transactions", {}).get(account_type.lower())[:limit]

    if not account_transactions:
        return f"No transactions found for {account_type}"

    result = f"Recent transaction for {account_type}:\n"
    for t in account_transactions:
        sign = "+" if t['amount'] > 0 else ""
        result += f"{t['date']} : {t['description']} ({sign}${t['amount']:,.2f})\n"

    return result

**A new idea here: default parameter values.** `limit: int = 5` means the LLM *may* supply a `limit` if the user says "show me my last 2 transactions," but if it doesn't, the tool still works with a sensible default. Type-annotated defaults like this are picked up by `@tool` and shown to the LLM as "optional" parameters.

> ⚠️ **Bug watch:** `user_data.get("transactions", {}).get(account_type.lower())[:limit]` slices the result of `.get(...)` **before** checking whether it's `None`. If `account_type` doesn't match any key (e.g. the same `"saving"` vs `"savings"` mismatch from before, or any typo), `.get(account_type.lower())` returns `None`, and `None[:limit]` raises a `TypeError: 'NoneType' object is not subscriptable` — **before** the `if not account_transactions:` check ever gets a chance to run. So the "friendly no-transactions-found message" is actually unreachable for a genuinely invalid `account_type`; instead the tool crashes with a raw Python exception. (This is exactly the kind of error the `handle_tool_errors` middleware in Section 8.3 is designed to catch and convert into a clean message — so functionally the user still gets *something* reasonable, but it's worth knowing the "clean" code path here is dead.)

Next, a tool that needs **only** the injected context and no LLM-supplied arguments at all.

In [ ]:
@tool
def get_personalized_greeting(runtime: ToolRuntime[UserContext]) -> str:
    """
        Get a personalized greeting for the user
    """
    name = runtime.context.user_name
    tier = runtime.context.membership_tier

    tier_benefits = {
        "basic": "You have access to standard features.",
        "premium": "As a premium member, you get priority support and advance analytics!",
        "platinum": "Welcome, platinum member! You have access to all featuers including personal advisor consultations.",
    }

    benefit_msg = tier_benefits.get(tier, "")
    return f"Hello, {name}! {benefit_msg}"

This is the simplest possible "context-only" tool — proof that a tool doesn't need *any* LLM-chosen arguments; it can exist purely to expose trusted, injected data in a formatted way.

Finally, the most complex tool in the file — one that **validates and mutates state**, and deliberately raises exceptions on bad input rather than returning error strings.

In [ ]:
@tool
def transfer_money(
    from_account: str,
    to_account: str,
    amount: float,
    runtime: ToolRuntime[UserContext]
) -> str:
    """
        Transfer money between accounts.

        Args:
            from_account : Source account ('checking','savings','investment')
            to_account : Destination account ('checking','savings','investment')
            amount: Amount to transfer (must be positive)
    """
    if amount <= 0:
        raise ValueError("Transfer amount must be positive")

    if amount > 10000:
        raise ValueError("Transfer amount exceeds the daily limit of 10,000")

    if from_account.lower() == to_account.lower():
        raise ValueError("Cannot transfer to the same account")

    user_id = runtime.context.user_id
    user_data = USER_DATABASE.get(user_id, {})
    accounts = user_data.get("accounts", {})

    from_balance = accounts.get(from_account.lower())

    if from_balance is None:
        raise ValueError(f"Source account {from_account} not found")

    if to_account is None:
        raise ValueError(f"Destination account {to_account} not found")

    if from_balance < amount:
        raise ValueError(f"Insufficient funds {from_account} balance: {from_balance:,.2f}")

    # Simulate successful transfer (a real implementation would mutate the DB / call an API here)
    return f"Successfully transferred ${amount: .2f} from {from_account} to {to_account}"

**Why `raise ValueError(...)` here, instead of `return "error: ..."` like the other tools did?** This is a genuine design choice, and it's the setup for the middleware you'll see in Section 8.3. Raising real Python exceptions means:
- the failure is unambiguous and can't be confused with a legitimate string result,
- exceptions can be caught and handled **generically, in one place** (the middleware), instead of every tool re-implementing its own "did this fail?" string convention,
- it plays well with Python's built-in exception hierarchy (`ValueError` for "the input was semantically wrong," which is exactly what all five checks above are).

This is a nice contrast to `get_account_balance`/`get_recent_transactions`, which handle their errors "locally" by returning descriptive strings. Both patterns are valid — the difference is **who is responsible for turning the error into something the LLM can read**. `transfer_money` delegates that responsibility upward, to middleware.

> ⚠️ **Bug watch (minor, cosmetic):** `if to_account is None` can never be true — `to_account` is typed as `str` and always comes from the LLM as a string (even an empty one, never `None`). The check that actually matters (does the destination account *exist* in `accounts`) is missing; as written, transferring to a nonexistent `to_account` (e.g. a typo like `"chekcing"`) will silently "succeed" and return a success message, even though no such account exists to receive the money. If this were real, this is the one bug in the file worth fixing first — it's a correctness issue with actual money, not just a UX rough edge. A fix would look like: `if to_account.lower() not in accounts: raise ValueError(...)`.

You now have all five tools. Let's collect them so the pattern is visible in one place before we move to middleware.

In [ ]:
all_tools = [
    get_account_balance,
    get_recent_transactions,
    calculate_budget,
    get_personalized_greeting,
    transfer_money,
]

for t in all_tools:
    print(f"{t.name:28s} -> {t.description.strip().splitlines()[0]}")

## 8. Middleware — behavior that wraps *around* the agent loop

Recall the loop from Section 0: *send messages to model → maybe call tools → repeat*. **Middleware** is code that hooks into specific points of that loop without you having to rewrite the loop itself. LangChain's agent middleware gives you (at least) three hook points, and this script uses all three:

| Decorator | Runs around... | Used here for |
|---|---|---|
| `@wrap_model_call` | every call to the LLM | picking *which model* to use, based on tier |
| `@dynamic_prompt` | prompt construction, before each model call | picking *which system prompt* to use, based on tier |
| `@wrap_tool_call` | every tool execution | catching exceptions raised by tools (like `transfer_money`'s `ValueError`s) and turning them into clean messages |

The common shape across all "wrap" middleware is the **handler pattern**: your function receives the original `request` *and* a `handler` callable that represents "the rest of the pipeline." You can inspect/modify the request, call `handler(request)` to continue, and inspect/modify the result — this is the same pattern used by middleware in web frameworks like Express or Django.

### 8.1 `@wrap_model_call` — routing to a model by tier

In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse

@wrap_model_call
def dynamic_model_selector(request: ModelRequest, handler) -> ModelResponse:
    """
        Selects model based on the membership tier
    """
    tier = request.runtime.context.membership_tier

    if tier == 'platinum':
        request = request.override(model=platinum_model)
        print(f"[Middleware] Using PLATINUM model (gpt-oss-120b)")
    elif tier == 'premium':
        request = request.override(model=premium_model)
        print(f"[MIDDLEWARE] Using PREMIUM model (llama-3.3-70b-versatile)")
    else:
        request = request.override(model=basic_model)
        print(f"[MIDDLEWARE] Using Basic model (llama-3.1-8b-instant)")

    return handler(request)

`request.override(model=...)` returns a *new* request object with the model swapped, leaving everything else (the conversation, the tools available, etc.) untouched — this immutable-update style (`override` returning a copy rather than mutating in place) is common in LangChain/LangGraph and makes middleware easier to reason about, since a middleware function can't accidentally corrupt shared state.

`handler(request)` is what actually continues the pipeline — **forgetting to call it (or to return its result) would silently swallow the entire model call**, so it's worth internalizing this as the one non-negotiable line in any `wrap_*` middleware you write.

> ⚠️ **Bug watch (cosmetic, but confusing):** the platinum branch's print statement says `"Using PLATINUM model (gpt-oss-120b)"`, but `platinum_model` is actually configured as `groq:llama-3.3-70b-versatile` — same model as premium, just without the `max_tokens` cap. `gpt-oss-120b` appears only in the commented-out `basic_model` block near the top of the file (lines 28-32) and was likely leftover from an earlier draft. Log messages like this lying to you during debugging is a classic source of wasted time — worth fixing the string to match reality.

### 8.2 `@dynamic_prompt` — building the system prompt per tier

Instead of one fixed system prompt, this middleware **constructs the prompt string at request time**, using the same `membership_tier` used above.

In [ ]:
from langchain.agents.middleware import dynamic_prompt

@dynamic_prompt
def tier_based_prompt(request: ModelRequest) -> str:
    """
    Generate system prompt based on user's membership tier
    """
    tier = request.runtime.context.membership_tier
    user_name = request.runtime.context.user_name
    base_prompt = f"""You are a personal finance assistant helping {user_name}.

    Your capabilities and when to use them:
    - Use `get_account_balance` ONLY when the user asks to CHECK a balance
    - Use `get_recent_transactions` when the user asks to SEE transactions
    - Use `calculate_budget` when the user asks about budgeting
    - Use `get_personalized_greeting` ONLY at the start of a conversation
    - Use `transfer_money` when the user says "transfer", "move money", or "send from X to Y"

    IMPORTANT
    - Report exactly what the tool returns
    - Call only neccssary tool. Dont call any other tool unneccssarily
    """

    if tier == "premium":
        return base_prompt + """

            PREMIUM MEMBER BENEFITS:
            - Provide helpful explanations with your responses
            - Offer occasional tips for financial improvement
            - Be friendly and informative
            - Balance detail with brevity"""

    elif tier == "platinum":
        return base_prompt + """

            PLATINUM MEMBER BENEFITS:
            - Provide detailed, comprehensive financial analysis
            - Offer proactive suggestions for wealth growth
            - Include market insights when relevant
            - Be thorough and consultative in your responses
            - Take extra time to explain complex concepts"""

    else:  # basic
        return base_prompt + """

            Guidelines:
            - Be concise and direct
            - Answer questions efficiently
            - Focus on the specific request
            - Keep responses brief but helpful"""

This is the single most important idea for building **multi-tenant** agents (agents serving many different kinds of users from one codebase): rather than writing three separate agents for three tiers, you write *one* agent whose behavior branches on context at exactly two points — which model runs (Section 8.1) and what instructions it's given (this section). Everything else — the tools, the code, the deployment — stays identical.

Notice the tool-usage rules baked into `base_prompt` (`"Use get_account_balance ONLY when..."`) — this is **prompt engineering as a substitute for hard constraints**. LangChain doesn't force the model to only call `get_personalized_greeting` at the start of a conversation; the prompt just *asks* it to behave that way. This is inherently a little unreliable (LLMs don't perfectly follow instructions) — for genuinely hard constraints (e.g. "never transfer more than $10,000") you want validation *inside the tool itself*, like `transfer_money` already does with its `raise ValueError` checks, not just a prompt instruction.

> ⚠️ **Bug watch (harmless but worth knowing):** `base_prompt = base_prompt = f"""..."""` assigns to `base_prompt` twice in one line — the middle `base_prompt =` is a no-op (Python allows chained assignment `a = b = value`, which normally sets *two different names* to the same value; here both sides happen to be the same name, so it's just redundant, not a bug in behavior). Almost certainly a copy-paste artifact. Doesn't change what the code does, but it's the kind of thing a linter would flag, and worth cleaning up so a reader doesn't pause on it wondering if something subtle is happening.

### 8.3 `@wrap_tool_call` — centralized error handling

This is where the `raise ValueError(...)` calls inside `transfer_money` (and the crash we identified inside `get_recent_transactions`) actually get caught.

In [ ]:
from langchain.agents.middleware import wrap_tool_call
from langchain.tools.tool_node import ToolCallRequest
from langchain_core.messages import ToolMessage

@wrap_tool_call
def handle_tool_errors(request: ToolCallRequest, handler) -> ToolMessage:
    """
        Gracefully handles tool execution errors.
    """
    tool_name = request.tool_call['name']

    try:
        return handler(request)
    except ValueError as e:
        error_message = f"{tool_name} failed: {str(e)}"
        print(f"[Error Handler] Caught ValueError: {e}")
        return ToolMessage(content=error_message, tool_call_id=request.tool_call["id"])

    except KeyError as e:
        error_message = f"{tool_name} error: Requried data not found {str(e)}"
        print(f"[Error Handler] Caught KeyError : {e}")
        return ToolMessage(content=error_message, tool_call_id=request.tool_call["id"])

    except Exception as e:
        error_message = f"{tool_name} encountered an error. Please try again or contact"
        print(f"[Error Handler] Caught Unexpected error: {type(e).__name__}-{e}")
        return ToolMessage(content=error_message, tool_call_id=request.tool_call["id"])

Two Python fundamentals worth calling out explicitly, since they matter a lot here:

- **Exception ordering matters.** `except ValueError` and `except KeyError` are listed *before* `except Exception`. Python checks `except` clauses top-to-bottom and uses the *first* one that matches. Since `ValueError` and `KeyError` are both more *specific* subclasses of the more *general* `Exception`, they have to come first — if `except Exception` were listed first, it would catch everything (including `ValueError`s) and the two specific branches below it would become **unreachable dead code**. This "specific before general" ordering is a rule you'll use constantly once you start writing your own error-handling code.
- **`ToolMessage`** is a LangChain message type that represents "the result of a tool call" in the conversation history, tagged with `tool_call_id` so the LLM can match it back to the specific tool call it made (this matters when multiple tools are called in parallel). Returning a `ToolMessage` from this middleware — instead of letting the exception propagate and crash the whole agent — means a tool failure becomes just another message the LLM gets to read and react to (e.g. "sorry, that transfer failed because ...") rather than a hard application crash.

Without this middleware, the `TypeError` bug identified in `get_recent_transactions`, or any of `transfer_money`'s `ValueError`s, would bubble all the way up and kill the whole `agent.invoke(...)` call. With it, the agent degrades gracefully — the *user experience* survives even when a *tool* fails, which is one of the biggest practical differences between a demo agent and a production-ready one.

**Middleware execution order.** All three middleware functions get passed to `create_agent(middleware=[...])` as a list (Section 9) — and list order matters, since it defines the order the hooks are chained in around the loop.

## 9. `ToolStrategy` — how "structured output" is actually enforced

You defined `FinancialResponse` back in Section 4, but a Pydantic class on its own doesn't *force* an LLM to use it — you need to tell the agent framework to enforce it. `ToolStrategy` is one such mechanism: it works by presenting the schema to the model **as if it were another tool** the model can "call" to submit its final answer, then validates that call's arguments against `FinancialResponse` and hands you back a real, validated Python object.

In [ ]:
from langchain.agents.structured_output import ToolStrategy

response_format = ToolStrategy(FinancialResponse)
# Later: create_agent(..., response_format=response_format)
# This means the agent's final reply is *forced* through the FinancialResponse
# schema -- you get response['structured_response'], a real FinancialResponse
# instance, not a string you'd have to parse yourself.

**Why route through a "fake tool call" instead of, say, asking the model to output raw JSON and parsing it yourself?** Because modern LLMs are specifically fine-tuned to produce well-formed function-call arguments (that's the whole "function calling" capability from Section 6), and that training tends to be far more reliable than asking a model to freeform-output valid JSON in a chat response, which is easy to get subtly wrong (a stray comment, a trailing comma, markdown code fences wrapping the JSON). `ToolStrategy` piggybacks on the more reliable capability.

There's a second, unused approach commented out at the very bottom of your script (inside the second, commented-out `main()`): calling `some_model.with_structured_output(FinancialResponse)` as a **separate**, tool-free model call that takes the agent's raw text answer and re-parses it into the schema afterward. That's a legitimate alternative design — "let the agent reason and answer freely, then have a second, cheap call extract structure from that answer" — with a different tradeoff: it costs an extra LLM call, but decouples "reasoning well" from "outputting valid structure," which can sometimes produce better *content* since the model isn't juggling both jobs in one pass. Worth knowing both patterns exist; `ToolStrategy` is the one actually wired into the live `agent`.

## 9.1 The dead `SYSTEM_PROMPT` string

Your file also defines a large, hand-written `SYSTEM_PROMPT` string (lines 392-413) with instructions like *"ALWAYS output the tool's response VERBATIM"* and *"Force the greeting function to run everytime."*

> ⚠️ **Bug watch:** in the actual `create_agent(...)` call (next section), `system_prompt=SYSTEM_PROMPT` is **commented out**:
> ```python
> #system_prompt=SYSTEM_PROMPT,
> ```
> So none of `SYSTEM_PROMPT`'s instructions are actually active — the *only* system prompt the agent ever sees is the one built dynamically by `tier_based_prompt` (Section 8.2), which is a **different, shorter set of instructions** that doesn't mention "verbatim" or "force the greeting." If you were debugging "why doesn't the agent always greet the user first," this is exactly the kind of silent mismatch (a big, carefully-written prompt sitting right there in the file, looking active, but never actually reaching the model) that costs the most debugging time. Worth either deleting `SYSTEM_PROMPT` entirely, or merging its useful instructions into `tier_based_prompt`'s `base_prompt`.

## 10. Putting it all together — `create_agent`

Every piece so far — the model(s), the tools, the context schema, the middleware, the structured output strategy — gets handed to a single factory call, `create_agent`, which builds the actual LangGraph agent object.

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

agent = create_agent(
    model=basic_model,
    tools=[
        get_account_balance,
        get_recent_transactions,
        calculate_budget,
        get_personalized_greeting,
        transfer_money
    ],
    # system_prompt=SYSTEM_PROMPT,   # <- intentionally left commented out, see 9.1
    context_schema=UserContext,
    middleware=[
        dynamic_model_selector,   # runs first: picks the model for this request
        tier_based_prompt,        # runs next: builds the prompt for the chosen tier
        handle_tool_errors        # wraps every individual tool call
    ],
    response_format=ToolStrategy(FinancialResponse),
    checkpointer=checkpointer
)

Walking through each argument, since every one of them is a concept you've now already built:

- **`model=basic_model`** — the *default/fallback* model. Note this gets **overridden per-request** by `dynamic_model_selector` (Section 8.1) — so in practice, which model actually answers depends on `context.membership_tier`, not on this argument. This default only matters if the middleware somehow didn't run.
- **`tools=[...]`** — the five functions from Section 7, now offered to the LLM as callable functions.
- **`context_schema=UserContext`** — tells the agent what shape of object to expect via the `context=` argument of `agent.invoke(...)`, and is what makes `ToolRuntime[UserContext]` inside tools, and `request.runtime.context` inside middleware, actually resolve to real data.
- **`middleware=[...]`** — the three hooks from Section 8, in a list. **Order matters**: `dynamic_model_selector` picks the model, then `tier_based_prompt` builds the prompt (both wrap the model call itself, but conceptually "select model" naturally happens first), and `handle_tool_errors` is a `wrap_tool_call` hook so it only fires around *tool* executions, independent of the other two.
- **`response_format=ToolStrategy(FinancialResponse)`** — forces the final answer through the schema from Section 4.
- **`checkpointer=checkpointer`** — see next.

### 10.1 The checkpointer — how the agent "remembers"

`InMemorySaver()` is a **checkpointer**: it saves the full conversation state (message history, intermediate tool results, etc.) after each step, keyed by a **thread ID**, so a *later* `agent.invoke(...)` call using the *same* thread ID continues the same conversation instead of starting fresh. "InMemory" means the storage is just process memory — it's perfect for development, but everything is lost when the Python process ends. A production deployment would swap this for a persistent checkpointer (e.g. backed by Postgres or SQLite) that survives restarts — same interface, different storage, again showing the "swap the implementation, keep the interface" pattern you've now seen twice (model providers in Section 2, and here).

> ⚠️ **Bug watch:** to actually *use* a checkpointer's memory, LangGraph expects you to pass a thread identifier via `config={"configurable": {"thread_id": "..."}}` on each `agent.invoke(...)` call, so it knows which saved conversation to load/append to. Looking at `main()` in the next section, **no `config`/`thread_id` is ever passed** to `agent.invoke(...)`. Practically, this means the `checkpointer` is wired up but never actually used to link separate calls into one conversation — each `agent.invoke(...)` call in `main()` behaves as an independent, fresh conversation regardless. This is very easy to miss because *nothing errors* — the agent runs fine, it's just silently not remembering anything across calls. If you want real multi-turn memory, the fix is to add a `thread_id` per user/session, e.g.:
> ```python
> config = {"configurable": {"thread_id": alice_context.user_id}}
> response = agent.invoke({...}, context=alice_context, config=config)
> ```

## 11. Running the agent — `main()`

Everything is wired up. Running the agent is now just: build a `UserContext` for whoever is asking, and call `agent.invoke(...)` with the conversation so far plus that context.

In [ ]:
def main():
    print("=" * 60)
    print("Stage 1: Simple Finance Assistant")
    print("=" * 60)

    alice_context = UserContext(
        user_id='user_001',
        user_name="Alice Johnson",
        membership_tier='platinum',
        preferred_currency="USD"
    )

    bob_context = UserContext(
        user_id="user_002",
        user_name="Bob smith",
        membership_tier="basic",
        preferred_currency="EUR"
    )

    balance_message = "What is my checking account balance?"
    print(f"Query:{balance_message}")

    response = agent.invoke(
        {'messages': [{'role': 'user', 'content': balance_message}]},
        context=bob_context
    )

    successful_transfer_prompt = "What's my financial situation? Check all my accounts and give me advice."
    print(f"\n Query : {successful_transfer_prompt}")

    response = agent.invoke(
        {"messages": [{'role': "user", "content": successful_transfer_prompt}]},
        context=alice_context
    )

    structured: FinancialResponse = response['structured_response']
    print("\n Structured Response")
    print(f"\n Summary: \n {structured.summary}")
    print(f"\n Details:\n {structured.details}")
    print(f"\n Action Items")
    for item in structured.action_items:
        print(f"*{item}")

    print("\n Warnings:")
    for warning in structured.warning:
        print(f"*{warning}")

    print(f"\n Confidence : \n {structured.confidence}")


if __name__ == "__main__":
    main()

A few things to connect back to earlier sections when you read this:

- **`{'messages': [{'role': 'user', 'content': ...}]}`** is the input shape `create_agent`-built agents expect: a dict with a `"messages"` key, holding a list of chat-message-shaped dicts. This is the same message format used across nearly all of LangChain's chat interfaces, so it'll look familiar in every other agent you build.
- **`context=bob_context`** is what makes `runtime.context` resolve correctly inside every tool and every middleware function for *this specific call* — swap in `alice_context` and the exact same tools/middleware behave differently (different model tier, different currency conversion, different prompt), without a single `if` statement in `main()` itself. That's the entire payoff of the context/middleware design from Sections 3, 7, and 8.
- **`response['structured_response']`** is only populated because of `response_format=ToolStrategy(FinancialResponse)` from Section 9 — without that, `response['messages'][-1].content` would just be a plain string you'd have to parse by hand (which is exactly what the second, commented-out `main()` at the bottom of the file does manually, as an illustration of the "harder way").
- Bob's query (checking balance, basic tier, EUR) never inspects `response` — its result is printed only via the `[MIDDLEWARE]`/`[Error Handler]` `print()` statements inside your middleware, not through `structured_response`. That's fine for a demo script, but note it means Bob's *actual answer text* is never shown to the console here — only Alice's structured response is.

**Try it yourself** (requires a real `GROQ_API_KEY` in a `.env` file next to this notebook):

In [ ]:
# Uncomment to actually run the agent end-to-end.
# Make sure a .env file with GROQ_API_KEY=... exists in this notebook's directory first.

# main()

## 12. Exercises — cement what you learned by fixing/extending the code

Try these, in order of difficulty. Each one exercises a different concept from above.

1. **(Docstring accuracy, Section 7)** Fix the `'saving'` / `'investement'` typos in `get_account_balance`'s docstring and error message so they match `USER_DATABASE`'s real keys (`'savings'`, `'investment'`).
2. **(Defensive coding, Section 7)** Fix `get_recent_transactions` so an invalid `account_type` returns the friendly `"No transactions found for..."` message instead of crashing with a `TypeError` — i.e. check for `None` *before* slicing.
3. **(Correctness, Section 7)** Fix `transfer_money` so it actually validates that `to_account` exists in the user's `accounts` dict (instead of the always-`False` `to_account is None` check).
4. **(New tool, Section 6)** Write a new `@tool` called `get_spending_by_category` that scans a user's `checking` transactions and buckets the negative amounts by rough category (you can hardcode a small keyword → category map, e.g. `"Grocery"` → `"food"`). This exercises the full pattern: `@tool`, `ToolRuntime[UserContext]`, docstring-as-LLM-instructions.
5. **(Middleware, Section 8)** Add a fourth tier, `"student"`, that (a) always uses `basic_model` regardless of what `dynamic_model_selector` currently checks, and (b) gets its own branch in `tier_based_prompt` with a friendlier, more educational tone.
6. **(Memory, Section 10.1)** Fix the missing `thread_id` bug: modify `main()` so that Alice's two hypothetical follow-up questions in a row actually share conversation memory via the checkpointer, and prove it by asking a second question that depends on the first (e.g. "what did I just ask you?").
7. **(Structured output, Section 9)** Add a new field to `FinancialResponse`, e.g. `risk_level: Literal["low", "medium", "high"]`, and update the tier-based prompts to explicitly ask premium/platinum users' responses to fill it in thoughtfully.

## Summary — the concepts you now have

- **`init_chat_model`**: provider-agnostic LLM loading, `"provider:model"` strings, `temperature`/`max_tokens`.
- **`@dataclass`**: lightweight typed structs for trusted, internal data (like request context).
- **`pydantic.BaseModel` + `Field` + `Literal`**: runtime-validated schemas, used both for forcing structured LLM output and (more generally) for validating any untrusted data.
- **`@tool`**: turning Python functions into LLM-callable capabilities; the docstring *is* the interface contract with the model.
- **`ToolRuntime[UserContext]`**: injecting trusted data into tools without exposing it to the LLM.
- **Middleware** (`wrap_model_call`, `dynamic_prompt`, `wrap_tool_call`): the handler pattern for hooking into an agent's loop — model selection, dynamic prompting, and centralized tool-error handling.
- **`ToolStrategy`**: enforcing structured final output via function-calling rather than free-text JSON.
- **Checkpointers + `thread_id`**: how conversational memory across multiple `invoke()` calls actually works (and how easy it is to wire up but forget to use).

If you want to go one level deeper next, the natural follow-ups are: (a) LangGraph's lower-level `StateGraph` API (what `create_agent` is built on top of, useful once you need control flow beyond a single agent loop), and (b) persistent checkpointers (Postgres/SQLite-backed) for when `InMemorySaver` needs to become production-durable.